# 06 · Circuit discovery and probe bundle (RQ1, H1)

Finds the attention heads and MLPs that carry the harmful→refusal signal using the matched pairs (clean = harmful, corrupt = harmless twin), validates them against random components (Table A), and writes the **probe bundle** that `check` (CIS) and `repair` use (docs/03 §4.1).

| | |
|---|---|
| **Kaggle settings** | Accelerator: GPU T4 · Internet: **On** · Secrets: `HF_TOKEN` |
| **Attach as inputs** | `safetypulse-lib` (sp_utils.py) · outputs of **03**, **04**, **05** |
| **Reads** | `pairs.jsonl`, `token_sets.json`, `direction/*` |
| **Writes** (`/kaggle/working/`) | `probe_bundle/` (probe_bundle.json, directions.safetensors, probe_pairs.jsonl) |

Keep this notebook and its outputs **private**: they contain harmful prompts.
After it finishes: *Save Version → Save & Run All*, then download the executed notebook into
`kaggle/executed/` and attach this notebook's output to the next notebook.

**Method.** CircuitKIT is the intended tool (cell below). Until its API is confirmed, we use the fallback from
docs/03 §5: *attribution patching* (a gradient estimate for every head and MLP), then *exact activation
patching* on the top candidates to confirm them, both at the last prompt token. Report which one produced the
final components.

In [ ]:
# Setup: find sp_utils.py (attach the `safetypulse-lib` dataset: see kaggle/README.md)
import glob, os, sys
_found = (glob.glob("/kaggle/input/**/sp_utils.py", recursive=True)
          + glob.glob("/kaggle/usr/lib/**/sp_utils.py", recursive=True)
          + glob.glob("../lib/sp_utils.py"))
assert _found, "sp_utils.py not found. Attach the safetypulse-lib dataset (Add Input) and re-run."
sys.path.insert(0, os.path.dirname(os.path.abspath(_found[0])))
import sp_utils as sp
sp.env_info()

In [ ]:
import random
import numpy as np, pandas as pd, torch
from safetensors.torch import load_file, save_file

MODEL = os.environ.get("SP_MODEL", sp.DEFAULT_MODEL)
SEED = 0
N_ATTR_PAIRS = 100 if not sp.SMOKE else 4    # calib pairs used for discovery
N_CANDIDATES = 40 if not sp.SMOKE else 6     # top attribution scores re-checked by exact patching
TOP_K = 20 if not sp.SMOKE else 3            # components kept in the circuit
N_RANDOM = 10 if not sp.SMOKE else 2
GEN_TOKENS = 64
OUT = sp.out_dir("probe_bundle")
sp.set_seed(SEED)
pairs = sp.load_jsonl(sp.find_input("pairs.jsonl"))
calib = [p for p in pairs if p["split"] == "calib"][:N_ATTR_PAIRS]
val = [p for p in pairs if p["split"] == "val"]
probe = [p for p in pairs if p["split"] == "probe"]
token_sets = sp.load_json(sp.find_input("token_sets.json"))[MODEL]
rd = sp.load_json(sp.find_input("refusal_direction.json"))
directions = load_file(str(sp.find_input("directions.safetensors")))["directions"]
LAYER = rd["layer"]
model, tok = sp.load_model(MODEL, dtype=torch.float32)

In [ ]:
# --- CircuitKIT (fill in once its API is known; see notebook 00) ---------------------------------
USE_CIRCUITKIT = False
if USE_CIRCUITKIT:
    raise NotImplementedError("call CircuitKIT discovery with clean=harmful, corrupt=harmless twin, "
                              "metric=refusal logit difference; map its output to 'attn_head:L:H' / 'mlp:L' ids")

In [ ]:
attr = sp.attribution_scores(model, tok, [p["harmful"] for p in calib], [p["harmless"] for p in calib], token_sets)
ranked = sorted(attr.items(), key=lambda kv: -kv[1])
candidates = [c for c, _ in ranked[:N_CANDIDATES]]
exact = sp.activation_patching_scores(model, tok, [p["harmful"] for p in calib], [p["harmless"] for p in calib],
                                      candidates, token_sets)
cand = pd.DataFrame({"attribution": {c: attr[c] for c in candidates}, "exact": exact}).sort_values("exact", ascending=False)
print("rank agreement (Spearman) attribution vs exact:", round(cand.corr(method="spearman").iloc[0, 1], 2))
components = [c for c in cand.index if cand.loc[c, "exact"] > 0][:TOP_K]
assert components, "no component raises refusal when patched: report this (gate G2 fallback: direction-only CIS)"
print(len(components), "components:", components)
cand.head(TOP_K)

## Validation: ablating the circuit vs. random components of the same size (Table A)

Components are **mean-ablated** (replaced by their mean over harmless calib prompts). Zero-ablating many whole MLPs breaks the model, so random sets also stop refusing (run 1: 4% vs 0%) and the control says nothing. `degenerate` is the share of incoherent answers: a refusal drop only counts if the text stays coherent.

In [ ]:
val_h = [p["harmful"] for p in val]
means = sp.component_means(model, tok, [p["harmless"] for p in calib])
samples = {}

def rates(name, handles=None):
    with sp.hooked(handles or []):
        out = sp.generate(model, tok, val_h, GEN_TOKENS, desc="val harmful")
    samples.setdefault(name, out[:5])
    return sp.refusal_rate(out), sp.degenerate_rate(out)

rng = random.Random(SEED)
rate_none, deg_none = rates("none")
rate_circ, deg_circ = rates("circuit (mean)", sp.ablate_components(model, components, means))
rate_zero, deg_zero = rates("circuit (zero)", sp.ablate_components(model, components))
rand = [rates("random (mean)", sp.ablate_components(model, sp.random_components_like(components, model, rng), means))
        for _ in range(N_RANDOM)]
rate_rand, deg_rand = [r for r, _ in rand], [d for _, d in rand]
table_a = rd["table_a"] + [
    {"ablation": f"discovered components, mean-ablated (k={len(components)})", "refusal_rate": rate_circ,
     "degenerate": deg_circ},
    {"ablation": f"random components, mean-ablated (k={len(components)}, n={N_RANDOM})",
     "refusal_rate": float(np.mean(rate_rand)), "std": float(np.std(rate_rand)), "degenerate": float(np.mean(deg_rand))},
    {"ablation": f"discovered components, zero-ablated (k={len(components)})", "refusal_rate": rate_zero,
     "degenerate": deg_zero},
]
for r in table_a:
    r["delta_vs_none"] = r["refusal_rate"] - rate_none
sp.save_json(samples, sp.out_dir("circuit_validation") / "samples.json")   # harmful completions: keep private
verdict = "PASS" if rate_circ < float(np.mean(rate_rand)) - 2 * float(np.std(rate_rand)) - 0.1 and deg_circ < 0.2 else "CHECK"
print(f"Circuit validation: {verdict}. circuit {rate_circ:.2f} (degenerate {deg_circ:.2f}) vs random "
      f"{np.mean(rate_rand):.2f} ± {np.std(rate_rand):.2f}; no ablation {rate_none:.2f} (degenerate {deg_none:.2f})")
pd.DataFrame(table_a)

## Probe bundle: base-model reference values on the probe set (so the base model scores CIS = 1.0)

In [ ]:
probe_h = [p["harmful"] for p in probe]
probe_s = [p["harmless"] for p in probe]
a_h = sp.resid_last(model, tok, probe_h)
a_s = sp.resid_last(model, tok, probe_s)
probe_diff = sp.mean_diff(a_h, a_s)
sep_base = sp.projection_separation(a_h[:, LAYER], a_s[:, LAYER], sp.unit(directions[LAYER]))
ld_h = sp.refusal_logit_diff(model, tok, probe_h, token_sets)
ld_s = sp.refusal_logit_diff(model, tok, probe_s, token_sets)
probe_attr = sp.attribution_scores(model, tok, probe_h, probe_s, token_sets)
comp_rows = [{"id": c, "type": sp.parse_component(c)[0], "layer": sp.parse_component(c)[1],
              "index": sp.parse_component(c)[2], "score": probe_attr[c], "discovery_exact": float(cand.loc[c, "exact"])}
             for c in components]
circ_base = sum(r["score"] for r in comp_rows)

save_file({"directions": directions.contiguous(), "probe_diff": probe_diff.contiguous()}, str(OUT / "directions.safetensors"))
probe_path = sp.save_jsonl(probe, OUT / "probe_pairs.jsonl")
bundle = {
    "schema_version": "1.0",
    "base_model": {"name": MODEL, "revision": None},
    "layer": LAYER,
    "direction_file": "directions.safetensors",
    "components": comp_rows,
    "discovery": {"method": "circuitkit" if USE_CIRCUITKIT else "attribution+exact patching (last token)",
                  "matched_pairs": pairs[0]["matched"], "n_pairs": len(calib)},
    "token_sets": {"refuse": token_sets["refuse"], "comply": token_sets["comply"]},
    "reference": {"sep_base": sep_base, "logit_gap_base": float(ld_h.mean() - ld_s.mean()), "circ_base": circ_base},
    "threshold": {"metric": None, "tau_warn": None, "tau_fail": None},
    "probe_set": {"path": probe_path.name, "sha256": sp.sha256_file(probe_path), "n_pairs": len(probe)},
    "table_a": table_a,
    "provenance": sp.provenance(SEED),
}
sp.save_json(bundle, OUT / "probe_bundle.json")
bundle["reference"]

In [ ]:
# Sanity check: the base model must score ~1.0 on every CIS variant.
bundle, tensors = sp.load_bundle(OUT)
check = sp.compute_cis(model, tok, probe, bundle, tensors)
print(check["scores"], f"{check['cost']['gpu_seconds']:.1f}s")
assert all(abs(v - 1) < 1e-3 for v in check["scores"].values()), "base model should score 1.0"